# Exercice 2 — Q-Learning sur FrozenLake-v1

## Étape 1 — Créer l'environnement et initialiser la Q-table

In [1]:
import gymnasium as gym
import numpy as np

env = gym.make("FrozenLake-v1")

print("Espace d'observation :", env.observation_space)
print("Espace d'action      :", env.action_space)

Espace d'observation : Discrete(16)
Espace d'action      : Discrete(4)


### La carte du lac

`S` = départ, `F` = glace (sûr), `H` = trou (perdu), `G` = but (récompense +1).  
L'état est le numéro de la case, de 0 (en haut à gauche) à 15 (en bas à droite).

In [2]:
carte = env.unwrapped.desc.astype(str)
n_lignes, n_colonnes = carte.shape

for i, ligne in enumerate(carte):
    etats = " ".join(f"{i * n_colonnes + j:>2}" for j in range(n_colonnes))
    print(" ".join(ligne), "   états :", etats)

print("\nActions : 0 = gauche, 1 = bas, 2 = droite, 3 = haut")

S F F F    états :  0  1  2  3
F H F H    états :  4  5  6  7
F F F H    états :  8  9 10 11
H F F G    états : 12 13 14 15

Actions : 0 = gauche, 1 = bas, 2 = droite, 3 = haut


### La Q-table

Une ligne par état, une colonne par action. `Q[s, a]` = estimation de la récompense future si on fait l'action `a` dans l'état `s`.

In [3]:
n_states = env.observation_space.n
n_actions = env.action_space.n

q_table = np.zeros((n_states, n_actions))

print("Dimensions de la Q-table :", q_table.shape)
q_table

Dimensions de la Q-table : (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

## Étape 2 — Boucle d'entraînement Q-Learning

### Hyperparamètres

In [4]:
n_episodes = 20_000

learning_rate = 0.1      # alpha : taille du pas de correction
discount_factor = 0.99   # gamma : importance des récompenses futures

epsilon = 1.0            # probabilité d'explorer (100 % au début)
epsilon_min = 0.01       # on garde toujours un peu d'exploration
epsilon_decay = 0.9995   # epsilon est multiplié par ce facteur après chaque épisode

### Entraînement

À chaque pas : choix epsilon-greedy, action, puis mise à jour de `q_table[state, action]` :

$$Q(s, a) \leftarrow Q(s, a) + \alpha \, \big[\, r + \gamma \max_{a'} Q(s', a') - Q(s, a) \,\big]$$

In [5]:
rng = np.random.default_rng(0)
env.action_space.seed(0)

q_table = np.zeros((n_states, n_actions))
rewards_per_episode = []

for episode in range(n_episodes):
    state, info = env.reset(seed=episode)
    total_reward = 0.0
    terminated = truncated = False

    while not (terminated or truncated):
        # Epsilon-greedy : explorer ou exploiter
        if rng.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(q_table[state, :])

        new_state, reward, terminated, truncated, info = env.step(action)

        # Mise à jour du Q-Learning
        old_value = q_table[state, action]
        max_future_q = np.max(q_table[new_state, :])
        q_table[state, action] = old_value + learning_rate * (
            reward + discount_factor * max_future_q - old_value
        )

        state = new_state
        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    rewards_per_episode.append(total_reward)

    if (episode + 1) % 2000 == 0:
        taux = np.mean(rewards_per_episode[-2000:])
        print(f"Épisode {episode + 1:>6} | epsilon = {epsilon:.3f} | taux de réussite (2000 derniers) = {taux:.1%}")

Épisode   2000 | epsilon = 0.368 | taux de réussite (2000 derniers) = 5.5%


Épisode   4000 | epsilon = 0.135 | taux de réussite (2000 derniers) = 19.9%


Épisode   6000 | epsilon = 0.050 | taux de réussite (2000 derniers) = 42.5%


Épisode   8000 | epsilon = 0.018 | taux de réussite (2000 derniers) = 59.2%


Épisode  10000 | epsilon = 0.010 | taux de réussite (2000 derniers) = 66.6%


Épisode  12000 | epsilon = 0.010 | taux de réussite (2000 derniers) = 69.1%


Épisode  14000 | epsilon = 0.010 | taux de réussite (2000 derniers) = 68.5%


Épisode  16000 | epsilon = 0.010 | taux de réussite (2000 derniers) = 69.3%


Épisode  18000 | epsilon = 0.010 | taux de réussite (2000 derniers) = 66.0%


Épisode  20000 | epsilon = 0.010 | taux de réussite (2000 derniers) = 68.7%


### La Q-table apprise

In [6]:
np.set_printoptions(precision=3, suppress=True)
print("Colonnes : gauche, bas, droite, haut\n")
print(q_table)

Colonnes : gauche, bas, droite, haut

[[0.555 0.5   0.492 0.503]
 [0.24  0.345 0.382 0.51 ]
 [0.41  0.415 0.402 0.471]
 [0.317 0.299 0.319 0.45 ]
 [0.569 0.469 0.369 0.451]
 [0.    0.    0.    0.   ]
 [0.225 0.093 0.391 0.143]
 [0.    0.    0.    0.   ]
 [0.3   0.452 0.303 0.609]
 [0.51  0.677 0.413 0.492]
 [0.638 0.398 0.373 0.275]
 [0.    0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.471 0.575 0.797 0.488]
 [0.709 0.88  0.763 0.694]
 [0.    0.    0.    0.   ]]


In [7]:
fleches = np.array(["←", "↓", "→", "↑"])
politique = fleches[np.argmax(q_table, axis=1)].reshape(carte.shape)
politique[carte == "H"] = "H"
politique[carte == "G"] = "G"

print("Meilleure action apprise dans chaque case :\n")
for ligne in politique:
    print(" ".join(ligne))

Meilleure action apprise dans chaque case :

← ↑ ↑ ↑
← H → H
↑ ↓ ← H
H → ↓ G


## Étape 3 — Évaluation de l'agent entraîné

L'agent n'explore plus (pas d'epsilon) et la Q-table n'est plus modifiée.

In [8]:
n_eval_episodes = 100
total_wins = 0

for episode in range(n_eval_episodes):
    # Graines différentes de celles de l'entraînement (0 à n_episodes - 1)
    state, info = env.reset(seed=n_episodes + episode)
    terminated = truncated = False

    while not (terminated or truncated):
        action = np.argmax(q_table[state, :])
        state, reward, terminated, truncated, info = env.step(action)

    if reward == 1.0:
        total_wins += 1

print(f"Taux de réussite sur {n_eval_episodes} épisodes : {total_wins / n_eval_episodes:.0%}")

Taux de réussite sur 100 épisodes : 69%


In [9]:
env.close()